# Blow-up detection diagnostic

For each variable, plots the spatial min/max time series and highlights
the window where exponential growth was detected (the segment fitted by
the log-linear regression).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

import sys
sys.path.insert(0, str(Path('..').resolve()))
from utils import VARS_LATEX, DEFAULT_VARIABLES
from metrics.blowup_time import detect_exponential_onset

Parameters

In [ ]:
model_name = 'Aurora'

# Local path — update to your file; not released in the repo
pred_path = Path(f'/path/to/pred_{model_name}.nc')

# Variables to inspect (names as stored in all_pred, level already in the name)
list_variables = DEFAULT_VARIABLES

# Physical plausibility bounds for clipping before blow-up detection. Calculated from 2 times the minimum/maximum of ERA5
# Format: {variable: [min_value, max_value]}
BOUNDS = {"2m_temperature": [90, 600], 
        "10m_u_component_of_wind": [-80, 75],
        "mean_sea_level_pressure": [45500, 210000],
        "geopotential_500": [22000, 118000],
        "temperature_500": [100, 550],
        "specific_humidity_500": [-0.0005, 0.025],
        "u_component_of_wind_300": [-140, 230],
        "temperature_100": [90, 480],
        "specific_humidity_100": [-0.0002, 0.0001]
        }

window_size  = 4 * 30   # steps (30 days at 6-hourly)
r2_threshold = 0.90   # minimum coefficient of determination
slope_min    = 0.001   # minimum slope

COLOR_MODEL = {
    'ERA5':         'black',
    'Aurora':       'tab:cyan',
    'SFNO':         'tab:green',
    'Pangu':        'tab:pink',
    'GraphCast':    'tab:purple',
    'FourCastNet':  'tab:olive',
    'FuXi':         'tab:brown',
    'AIFS':         'tab:red',
    'GenCast':      'tab:orange',
}

In [ ]:
# all_pred: (time, latitude, longitude), level already dropped
ds_pred = xr.open_dataset(pred_path)

In [ ]:
color = COLOR_MODEL.get(model_name, 'tab:blue')
X = np.arange(window_size).reshape(-1, 1)

for var in list_variables:
    if var not in ds_pred:
        print(f'{var}: not found in all_pred, skipping.')
        continue

    pred = ds_pred[var]

    # Rolling spatial min / max
    p_min = pred.min(dim=['latitude', 'longitude']).compute().rolling(time=16, center=True).mean()
    p_max = pred.max(dim=['latitude', 'longitude']).compute().rolling(time=16, center=True).mean()

    # Clip to physical bounds if defined
    if var in BOUNDS:
        lo, hi = BOUNDS[var]
        p_min = xr.where(p_min < lo, lo, xr.where(p_min > hi, hi, p_min))
        p_max = xr.where(p_max < lo, lo, xr.where(p_max > hi, hi, p_max))

    # Detect onset
    log_min, it_exp_min = detect_exponential_onset(
        p_min.values, window_size=window_size,
        r2_threshold=r2_threshold, slope_min=slope_min,
    )
    log_max, it_exp_max = detect_exponential_onset(
        p_max.values, window_size=window_size,
        r2_threshold=r2_threshold, slope_min=slope_min,
    )

    if it_exp_min is None and it_exp_max is None:
        print(f'{VARS_LATEX.get(var, var)}: no blow-up detected')
        continue

    fig, ax = plt.subplots(figsize=(9, 3.5))

    ax.fill_between(
        np.arange(pred.time.size),
        pred.min(dim=['latitude', 'longitude']).values,
        pred.max(dim=['latitude', 'longitude']).values,
        color='k', alpha=0.12, label='Spatial min–max',
    )
    ax.plot(
        pred.mean(dim=['latitude', 'longitude']).values,
        color='k', lw=0.8, label='Spatial mean',
    )

    if it_exp_min is not None:
        ax.plot(p_min.values, color='tab:red', alpha=0.35, lw=0.8,
                label='Rolling spatial min')
        ax.plot(
            np.arange(it_exp_min, it_exp_min + window_size),
            p_min.values[it_exp_min : it_exp_min + window_size],
            'r--', lw=2, label=f'Onset (min) → step {it_exp_min}',
        )

    if it_exp_max is not None:
        ax.plot(p_max.values, color='tab:blue', alpha=0.35, lw=0.8,
                label='Rolling spatial max')
        ax.plot(
            np.arange(it_exp_max, it_exp_max + window_size),
            p_max.values[it_exp_max : it_exp_max + window_size],
            'b--', lw=2, label=f'Onset (max) → step {it_exp_max}',
        )

    it_exp = min((v for v in [it_exp_min, it_exp_max] if v is not None), default=None)
    blowup_days = it_exp / 4 if it_exp is not None else None
    title_suffix = f'blow-up at {blowup_days:.0f} days' if blowup_days else 'no blow-up'

    if var in BOUNDS:
        ax.set_ylim(BOUNDS[var])
    ax.set_xlabel('Rollout steps (6-hourly)')
    ax.set_ylabel(var.replace('_', ' '))
    ax.set_title(f'{VARS_LATEX.get(var, var)} — {model_name} — {title_suffix}')
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    plt.show()